   
# ZeroBus Ingest Endpoint Validation

End-to-end validation of the dbxWearables ZeroBus AppKit gateway:
1. Health check (ZeroBus env vars configured)
2. POST realistic test payloads for all 5 HealthKit record types
3. Verify records land in bronze table with correct `source_platform`, `user_id`, and header capture

**App:** `dev-dbxw-0bus-ingest` | **Table:** `{catalog}.{schema}.wearables_zerobus`

In [0]:
dbutils.widgets.text("app_base_url", "https://dev-dbxw-0bus-ingest-7474657291520070.aws.databricksapps.com", "App Base URL")
dbutils.widgets.text("catalog_use", "hls_fde_dev", "Catalog")
dbutils.widgets.text("schema_use", "dev_matthew_giglia_wearables", "Schema")

APP_BASE_URL = dbutils.widgets.get("app_base_url").rstrip("/")
CATALOG = dbutils.widgets.get("catalog_use")
SCHEMA = dbutils.widgets.get("schema_use")

print(f"App Base URL : {APP_BASE_URL}")
print(f"Catalog      : {CATALOG}")
print(f"Schema       : {SCHEMA}")

In [0]:
import requests
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── Authenticate to Databricks App (audience-scoped token exchange) ───────────
# Ref: https://docs.databricks.com/aws/en/dev-tools/databricks-apps/connect-local/

# Extract app name from URL: https://<app-name>-<workspace-id>.<region>.databricksapps.com
hostname = APP_BASE_URL.split("//")[1].split(".")[0]
parts = hostname.rsplit("-", 1)
app_name = parts[0] if parts[-1].isdigit() else hostname

# Get app's OAuth client ID
app_details = w.apps.get(app_name)
app_client_id = app_details.oauth2_app_client_id

# Exchange notebook token for audience-scoped access token
notebook_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

token_resp = requests.post(
    url=f"{w.config.host}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)
assert token_resp.status_code == 200, f"Token exchange failed: {token_resp.text}"

AUTH_HEADERS = {"Authorization": f"Bearer {token_resp.json()['access_token']}"}

# Define endpoints
HEALTH_URL = f"{APP_BASE_URL}/api/v1/healthkit/health"
INGEST_URL = f"{APP_BASE_URL}/api/v1/healthkit/ingest"
TABLE_FQN  = f"{CATALOG}.{SCHEMA}.wearables_zerobus"

print(f"App: {app_name}")
print(f"Auth: ✅ audience-scoped token acquired")
print(f"Table: {TABLE_FQN}")

In [0]:
resp = requests.get(HEALTH_URL, headers=AUTH_HEADERS, timeout=15)
data = resp.json()

assert resp.status_code == 200, f"Health check failed: {resp.status_code}"
assert data.get("env_configured") is True, f"ZeroBus env vars missing: {data.get('missing_env_vars')}"

print(f"✅ Health check passed")
print(f"   status: {data['status']}")
print(f"   env_configured: {data['env_configured']}")
print(f"   target_table: {data['target_table']}")

In [0]:
import random
from datetime import datetime, timezone, timedelta
from uuid import uuid4

def uuid() -> str:
    """Generate uppercase UUID string matching HealthKit format."""
    return str(uuid4()).upper()

def iso(dt: datetime) -> str:
    """Format datetime as ISO 8601 UTC string."""
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

# Base time: now (UTC), rounded to nearest minute
now = datetime.now(timezone.utc).replace(second=0, microsecond=0)
today = now.date().isoformat()

# ── Realistic value distributions ────────────────────────────────────────────

def heart_rate() -> float:
    """Resting HR 55-85, occasional elevated 90-140."""
    return round(random.gauss(72, 12), 1) if random.random() > 0.2 else round(random.uniform(90, 140), 1)

def step_count(duration_hours: float = 1.0) -> float:
    """~4000-8000 steps/hour walking, varies widely."""
    base = random.gauss(5500, 1500) * duration_hours
    return round(max(100, base), 0)

def workout_duration_min() -> float:
    """Most workouts 20-60 min, some longer."""
    return round(random.triangular(15, 90, 35), 0)

def calories_from_duration(duration_min: float, intensity: float = 1.0) -> float:
    """~8-12 kcal/min for moderate activity."""
    return round(duration_min * random.uniform(8, 12) * intensity, 1)

def distance_from_duration(duration_min: float, activity: str) -> float:
    """Pace varies by activity type."""
    pace_m_per_min = {"running": 180, "walking": 85, "cycling": 350, "swimming": 50}.get(activity, 100)
    return round(duration_min * pace_m_per_min * random.uniform(0.8, 1.2), 0)

def sleep_hours() -> float:
    """6.5-8.5 hours typical, occasionally shorter/longer."""
    return round(random.triangular(5.5, 9.5, 7.5), 2)

# ── Generate realistic payloads ──────────────────────────────────────────────

# Samples: 2 heart rate readings + 1 step count
hr1_time = now - timedelta(hours=random.uniform(1, 4))
hr2_time = now - timedelta(hours=random.uniform(0.1, 1))
steps_start = now - timedelta(hours=random.uniform(2, 5))
steps_duration_hr = random.uniform(0.5, 2)

samples = [
    {
        "uuid": uuid(),
        "type": "HKQuantityTypeIdentifierHeartRate",
        "value": heart_rate(),
        "unit": "count/min",
        "start_date": iso(hr1_time),
        "end_date": iso(hr1_time),
        "source_name": "Apple Watch",
        "source_bundle_id": f"com.apple.health.{uuid()}",
        "metadata": {"HKMetadataKeyHeartRateMotionContext": str(random.choice([0, 1, 2]))},
    },
    {
        "uuid": uuid(),
        "type": "HKQuantityTypeIdentifierHeartRate",
        "value": heart_rate(),
        "unit": "count/min",
        "start_date": iso(hr2_time),
        "end_date": iso(hr2_time),
        "source_name": "Apple Watch",
        "source_bundle_id": f"com.apple.health.{uuid()}",
        "metadata": None,
    },
    {
        "uuid": uuid(),
        "type": "HKQuantityTypeIdentifierStepCount",
        "value": step_count(steps_duration_hr),
        "unit": "count",
        "start_date": iso(steps_start),
        "end_date": iso(steps_start + timedelta(hours=steps_duration_hr)),
        "source_name": random.choice(["iPhone", "Apple Watch"]),
        "source_bundle_id": "com.apple.health",
        "metadata": None,
    },
]

# Workout: 1 running session earlier today
workout_start = now - timedelta(hours=random.uniform(4, 10))
workout_dur_min = workout_duration_min()
workout_activity = random.choice(["running", "walking", "cycling"])

workouts = [
    {
        "uuid": uuid(),
        "activity_type": workout_activity,
        "activity_type_raw": {"running": 37, "walking": 52, "cycling": 13}.get(workout_activity, 0),
        "start_date": iso(workout_start),
        "end_date": iso(workout_start + timedelta(minutes=workout_dur_min)),
        "duration_seconds": workout_dur_min * 60,
        "total_energy_burned_kcal": calories_from_duration(workout_dur_min),
        "total_distance_meters": distance_from_duration(workout_dur_min, workout_activity),
        "source_name": "Apple Watch",
        "metadata": None,
    },
]

# Sleep: last night's sleep session with realistic stages
sleep_hr = sleep_hours()
sleep_end = now.replace(hour=random.randint(5, 8), minute=random.randint(0, 59)) - timedelta(days=0 if now.hour > 10 else 0)
sleep_start = sleep_end - timedelta(hours=sleep_hr)

# Stage breakdown: ~5% awake, ~50% core, ~20% deep, ~25% REM
stage_fracs = {"awake": 0.05, "asleepCore": 0.50, "asleepDeep": 0.20, "asleepREM": 0.25}
stages = []
cursor = sleep_start
for stage_name, frac in stage_fracs.items():
    stage_dur = timedelta(hours=sleep_hr * frac)
    stages.append({
        "uuid": uuid(),
        "stage": stage_name,
        "start_date": iso(cursor),
        "end_date": iso(cursor + stage_dur),
    })
    cursor += stage_dur

sleep = [
    {
        "start_date": iso(sleep_start),
        "end_date": iso(sleep_end),
        "stages": stages,
    },
]

# Activity summary: yesterday's rings
active_cal = round(random.triangular(300, 700, 480), 1)
exercise_min = round(random.triangular(15, 60, 32), 0)
stand_hr = random.randint(8, 14)

activity_summaries = [
    {
        "date": (now - timedelta(days=1)).date().isoformat(),
        "active_energy_burned_kcal": active_cal,
        "active_energy_burned_goal_kcal": random.choice([400.0, 500.0, 600.0]),
        "exercise_minutes": exercise_min,
        "exercise_minutes_goal": 30.0,
        "stand_hours": stand_hr,
        "stand_hours_goal": 12,
    },
]

# Deletes: reference one of the sample UUIDs (simulates user deleting data)
deletes = [
    {
        "uuid": samples[0]["uuid"],  # Reference the first HR sample
        "sample_type": "HKQuantityTypeIdentifierHeartRate",
    },
]

# ── Assemble PAYLOADS dict ───────────────────────────────────────────────────
PAYLOADS = {
    "samples": samples,
    "workouts": workouts,
    "sleep": sleep,
    "activity_summaries": activity_summaries,
    "deletes": deletes,
}

def to_ndjson(records: list) -> str:
    """Serialize a list of dicts to NDJSON (one JSON object per line)."""
    return "\n".join(json.dumps(r) for r in records)

# Print summary
print(f"Generated {sum(len(v) for v in PAYLOADS.values())} realistic test records:")
for rt, recs in PAYLOADS.items():
    print(f"  {rt:20s} {len(recs)} record(s)")
print(f"\nSample UUIDs (unique per run):")
for s in samples[:2]:
    print(f"  {s['type'].split('Identifier')[-1]}: {s['uuid']}")

In [0]:
from datetime import datetime, timezone

INGESTED_RECORD_IDS = []  # Collect for downstream verification
results = {}
failures = []

for record_type, records in PAYLOADS.items():
    headers = {
        **AUTH_HEADERS,
        "Content-Type": "application/x-ndjson",
        "X-Record-Type": record_type,
        "X-Platform": "apple_healthkit",
        "X-App-Version": "1.0.0",
        "X-Upload-Timestamp": datetime.now(timezone.utc).isoformat(),
        "X-Device-Id": "validation-notebook",
    }
    
    resp = requests.post(INGEST_URL, headers=headers, data=to_ndjson(records), timeout=30)
    resp_json = resp.json() if resp.status_code == 200 else {}
    
    status = "✅" if resp.status_code == 200 else "❌"
    print(f"{status} {record_type:20s} {len(records)} rec(s) → {resp_json.get('records_ingested', 0)} ingested ({resp_json.get('duration_ms', 0)}ms)")
    
    if resp.status_code == 200:
        INGESTED_RECORD_IDS.extend(resp_json.get("record_ids", []))
        results[record_type] = resp_json
    else:
        failures.append(f"{record_type}: {resp.status_code}")
        print(f"   Error: {resp.text[:200]}")

print(f"\n{'='*60}")
if failures:
    raise AssertionError(f"Ingest failed for: {', '.join(failures)}")
print(f"✅ All {len(PAYLOADS)} record types ingested — {len(INGESTED_RECORD_IDS)} total records")
print(f"   Record IDs stored in INGESTED_RECORD_IDS for verification")

In [0]:
from pyspark.sql.functions import col, lit

print("="*70)
print("VERIFICATION")
print("="*70)

# ── Expected user_id: current notebook runner ────────────────────────
EXPECTED_USER_ID = spark.sql("SELECT current_user()").first()[0]
print(f"\nExpected user_id: {EXPECTED_USER_ID}")

validations_passed = 0
validations_failed = 0

# ── 1. Verify ingested records exist in table ───────────────────────────
print("\n1. Record IDs from POST exist in table:")

if INGESTED_RECORD_IDS:
    ids_list = ", ".join([f"'{rid}'" for rid in INGESTED_RECORD_IDS])
    df_ingested = spark.sql(f"""
        SELECT record_id, record_type, source_platform, user_id, ingested_at
        FROM {TABLE_FQN}
        WHERE record_id IN ({ids_list})
    """)
    found_count = df_ingested.count()
    expected_count = len(INGESTED_RECORD_IDS)
    
    if found_count == expected_count:
        print(f"   ✅ All {expected_count} records found in table")
        validations_passed += 1
    else:
        print(f"   ❌ Expected {expected_count}, found {found_count}")
        validations_failed += 1
else:
    print("   ⚠️  No record IDs to verify (INGESTED_RECORD_IDS is empty)")
    validations_failed += 1

# ── 2. Verify source_platform populated ────────────────────────────────
print("\n2. source_platform column populated:")

if INGESTED_RECORD_IDS:
    platforms = df_ingested.select("source_platform").distinct().collect()
    platform_values = [r["source_platform"] for r in platforms]
    
    if "apple_healthkit" in platform_values and len(platform_values) == 1:
        print(f"   ✅ All records have source_platform='apple_healthkit'")
        validations_passed += 1
    elif None in platform_values or "unknown" in platform_values:
        print(f"   ❌ Some records have NULL or 'unknown' source_platform: {platform_values}")
        validations_failed += 1
    else:
        print(f"   ⚠️  Unexpected platform values: {platform_values}")
        validations_failed += 1

# ── 3. Verify header blocklist (no authorization header stored) ─────────
print("\n3. Header blocklist (authorization stripped):")

if INGESTED_RECORD_IDS:
    sample_row = spark.sql(f"""
        SELECT headers::STRING AS headers_json
        FROM {TABLE_FQN}
        WHERE record_id = '{INGESTED_RECORD_IDS[0]}'
    """).collect()[0]
    
    headers_dict = json.loads(sample_row["headers_json"])
    
    blocklisted = [h for h in ["authorization", "cookie", "set-cookie"] if h in headers_dict]
    expected_headers = ["x-record-type", "x-platform", "x-device-id", "content-type"]
    present_headers = [h for h in expected_headers if h in headers_dict]
    
    if not blocklisted and len(present_headers) >= 3:
        print(f"   ✅ No blocklisted headers, {len(headers_dict)} headers captured")
        print(f"      Present: {', '.join(list(headers_dict.keys())[:6])}...")
        validations_passed += 1
    else:
        print(f"   ❌ Blocklisted headers found: {blocklisted}" if blocklisted else f"   ❌ Missing expected headers")
        validations_failed += 1

# ── 4. Verify VARIANT body round-trip ──────────────────────────────────
print("\n4. VARIANT body round-trip:")

if INGESTED_RECORD_IDS:
    samples_ids = results.get("samples", {}).get("record_ids", [])
    if samples_ids:
        body_row = spark.sql(f"""
            SELECT body::STRING AS body_json, record_type
            FROM {TABLE_FQN}
            WHERE record_id = '{samples_ids[0]}'
        """).collect()[0]
        
        body_dict = json.loads(body_row["body_json"])
        expected_keys = {"uuid", "type", "value", "unit", "start_date", "end_date", "source_name"}
        present_keys = set(body_dict.keys())
        
        if expected_keys.issubset(present_keys):
            print(f"   ✅ VARIANT body parsed correctly ({len(body_dict)} keys)")
            print(f"      Sample: uuid={body_dict['uuid'][:8]}..., value={body_dict['value']}")
            validations_passed += 1
        else:
            missing = expected_keys - present_keys
            print(f"   ❌ Missing keys in body: {missing}")
            validations_failed += 1
    else:
        print("   ⚠️  No samples records to verify")

# ── 5. Verify all record types present ─────────────────────────────────
print("\n5. All 5 record types present in table:")

df_types = spark.sql(f"""
    SELECT record_type, COUNT(*) AS count
    FROM {TABLE_FQN}
    GROUP BY record_type
    ORDER BY record_type
""")

type_counts = {r["record_type"]: r["count"] for r in df_types.collect()}
expected_types = set(PAYLOADS.keys())
present_types = set(type_counts.keys())

if expected_types.issubset(present_types):
    print(f"   ✅ All 5 record types found:")
    for rt in sorted(expected_types):
        print(f"      {rt:20s} {type_counts.get(rt, 0):>5} rows")
    validations_passed += 1
else:
    missing_types = expected_types - present_types
    print(f"   ❌ Missing record types: {missing_types}")
    validations_failed += 1

# ── 6. Verify user_id matches current user ─────────────────────────────
print(f"\n6. user_id = '{EXPECTED_USER_ID}':")

if INGESTED_RECORD_IDS:
    user_ids = df_ingested.select("user_id").distinct().collect()
    user_id_values = [r["user_id"] for r in user_ids]
    
    if len(user_id_values) == 1 and user_id_values[0] == EXPECTED_USER_ID:
        print(f"   ✅ All records have user_id='{EXPECTED_USER_ID}'")
        validations_passed += 1
    else:
        print(f"   ❌ Expected user_id='{EXPECTED_USER_ID}', found: {user_id_values}")
        validations_failed += 1

# ── SUMMARY ────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"\n   Validations passed: {validations_passed}")
print(f"   Validations failed: {validations_failed}")
print()

if validations_failed == 0:
    print("✅ ALL VALIDATIONS PASSED — ZeroBus ingest pipeline is working correctly")
else:
    print(f"❌ {validations_failed} VALIDATION(S) FAILED — review errors above")
    raise AssertionError(f"{validations_failed} validation(s) failed")

print("\n" + "="*70)